In [1]:
using Plots, JLD2, LinearAlgebra, BSON

# Load the Generic Problem Interface and Custom Method Solvers
include("src/ProblemInterface.jl")
include("src/GeneralPINN.jl")
include("src/GeneralBSDE.jl")
include("src/GeneralADP.jl")
include("src/GeneralNVI.jl")

using .ProblemInterface
using .GeneralPINN
using .GeneralADP
using .GeneralBSDE
using .GeneralNVI

In [7]:
const α_param = 0.5f0
const r_param = 3.0f0
const σ_param = 0.1f0
const X_MAX   = 15.0f0

prob = StochasticProblem(
    0.05f0, # rho (Discount rate)
    1,    # x_dim
    1,    # u_dim
    
    # Reward function c(x, u)
    (x, u) -> 1.0f0 - exp(-α_param * u[1]),
    
    # Drift function f(x, u)
    (x, u) -> [r_param * x[1] - u[1]],
    
    # Diffusion matrix σ(x)
    (x) -> reshape([σ_param * x[1]], 1, 1),
    
    # Discrete step dynamics (Euler-Maruyama with absorbing boundary at 0)
    (x, u, dt) -> begin
        if x[1] <= 0.0f0 return [0.0f0] end
        dx = (r_param * x[1] - u[1]) * dt + (σ_param * x[1]) * Float32(sqrt(dt)) * randn(Float32)
        return [max(0.0f0, x[1] + dx)]
    end,
    
    # Analytical optimal policy u*(∇V, x)
    (grad_V, x) -> begin
        dV_dx = grad_V[1]
        if dV_dx > 0.0f0 && (dV_dx / α_param) < 1.0f0
            return [- (1.0f0 / α_param) * log(dV_dx / α_param)]
        else
            return [0.0f0]
        end
    end,
    
    # State space uniform sampler (Float32 native)
    (num_samples) -> [rand(Float32, 1) .* X_MAX for _ in 1:num_samples],
    
    # PINN boundary condition penalty: V(0) = 0
    (model) -> model([0.0f0])[1]^2
)

results_jld2 = JLD2.load("results/asset_harvest_data.jld2")
# results_bson = BSON.load("results/asset_harvest_models.bson")

Dict{String, Any} with 9 entries:
  "v_nvi"  => Float32[644.766, 694.181, 743.596, 793.012, 842.427, 891.843, 941…
  "u_pinn" => Float32[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0,…
  "u_adp"  => Float32[2.22817, 2.19268, 2.15759, 2.12292, 2.08865, 2.05491, 2.0…
  "v_true" => [0.00344828, 0.029423, 0.0553977, 0.0813724, 0.107347, 0.133322, …
  "v_pinn" => Float32[0.0287277, 0.015207, -0.030459, -0.145205, -0.328826, -0.…
  "v_bsde" => Float32[0.0173998, 0.0523428, 0.0863125, 0.120282, 0.154252, 0.18…
  "u_true" => [0.743127, 0.743127, 0.743127, 0.743127, 0.743127, 0.743127, 0.74…
  "v_adp"  => Float32[3.86615, 3.88761, 3.90907, 3.93053, 3.95199, 3.97345, 3.9…
  "u_nvi"  => Float32[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0,…

In [8]:
results_bson = BSON.load("results/asset_harvest_models.bson")

UndefVarError: UndefVarError: `#26#27` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [9]:
# Save each variable in the results dictionary to the global scope
for (k, v) in pairs(results_jld2)
    name = k isa Symbol ? k : Symbol(k)
    @eval Main $(name) = $(v)
end

In [10]:
grid_points = 200
x_test_raw  = collect(range(0.01f0, X_MAX, length=grid_points))
x_test      = [[x] for x in x_test_raw]

# --- 5.1 Compute Mathematically Exact Harvesting Baselines ---
const slope_true = α_param / (α_param * r_param - prob.rho)
v_true = [slope_true * x for x in x_test_raw]

const constant_u = -(1.0f0 / α_param) * log(1.0f0 / (α_param * r_param - prob.rho))
u_true = [constant_u for _ in x_test_raw]


# --- 5.3 Performance Reporting (RMSE) ---
rmse(pred, true_val) = sqrt(sum((pred .- true_val).^2) / length(true_val))

println("\n==================================================")
println("   ASSET HARVESTING QUANTITATIVE METRICS (RMSE)   ")
println("==================================================")
println("PINN / DGM RMSE:            ", rmse(v_pinn, v_true))
println("Deep BSDE RMSE:             ", rmse(v_bsde, v_true))
println("ADP (Actor-Critic) RMSE:    ", rmse(v_adp, v_true))
println("Neural Value Iteration RMSE:", rmse(v_nvi, v_true))
println("==================================================")


   ASSET HARVESTING QUANTITATIVE METRICS (RMSE)   
PINN / DGM RMSE:            3.297231696742289
Deep BSDE RMSE:             0.9326539788668299
ADP (Actor-Critic) RMSE:    2.0840993980664195
Neural Value Iteration RMSE:6247.683963408133


In [11]:
plt_val = plot(x_test_raw, v_true, label="Ground Truth (Analytical)", linewidth=3, color=:black)
plot!(plt_val, x_test_raw, v_pinn, label="PINN / DGM", linewidth=2.0, color=:blue)
plot!(plt_val, x_test_raw, v_bsde, label="Deep BSDE (Truncated)", linewidth=2.0, color=:green, linestyle=:dash)
plot!(plt_val, x_test_raw, v_adp, label="ADP (Actor-Critic)", linewidth=2.0, color=:red, linestyle=:dot)
plot!(plt_val, x_test_raw, v_nvi, label="Neural Value Iteration", linewidth=2.0, color=:purple, linestyle=:dashdot)

title!(plt_val, "Value Function V(x) - Asset Harvesting")
xlabel!(plt_val, "Resource Population Size (x)")
ylabel!(plt_val, "Expected Discounted Harvest V(x)")
plot!(plt_val, legend=:bottomright, grid=true)

# --- 5.5 Plot 2: Implied Policy Optimization Comparison ---
function extract_policy(model, x_vec)
    et = eltype(x_vec)
    h = convert(et, 1e-4)
    v_up = model(x_vec .+ h)[1]
    x_minus = x_vec .- h
    x_minus_clamped = [ max(xx, zero(et)) for xx in x_minus ]
    v_dn = model(x_minus_clamped)[1]
    grad = [(v_up - v_dn) / (2 * h)]
    return prob.optimal_control_law(grad, x_vec)[1]
end

u_pinn = [extract_policy(pinn_model, x) for x in x_test]
u_adp  = [max(0.0f0, adp_actor(x)[1]) for x in x_test] 
u_nvi  = [extract_policy(nvi_active, x) for x in x_test]

plt_pol = plot(x_test_raw, u_true, label="Ground Truth Policy", linewidth=3, color=:black)
plot!(plt_pol, x_test_raw, u_pinn, label="PINN Induced Policy", linewidth=2.0, color=:blue)
plot!(plt_pol, x_test_raw, u_adp, label="ADP Explicit Actor", linewidth=2.0, color=:red, linestyle=:dot)
plot!(plt_pol, x_test_raw, u_nvi, label="NVI Induced Policy", linewidth=2.0, color=:purple, linestyle=:dashdot)

title!(plt_pol, "Optimal Harvesting Policy u*(x)")
xlabel!(plt_pol, "Resource Population Size (x)")
ylabel!(plt_pol, "Harvesting Extraction Rate (u)")
plot!(plt_pol, legend=:topleft, grid=true)

master_plot = plot(plt_val, plt_pol, layout=(2,1), size=(800, 700))

UndefVarError: UndefVarError: `pinn_model` not defined in `Main`
Suggestion: check for spelling errors or missing imports.